# 02 · Topologia do grafo de memória

As faces **não** são declaradas — são as órbitas de φ = σ∘α. Este notebook
mede o que a política de inserção σ realmente produziu.

Leia junto com `docs/COERENCIA.md`:

* **C1** — gênero por componente conexa (`V − E + F = 2C − 2g`);
* **C3** — faces-folha vs bígonos genuínos;
* **C9** — o risco de degenerar em pouquíssimas faces gigantes.


In [ ]:
from nbutils import *

# results/ por padrão; use setup(dry=True) para inspecionar um smoke run offline
ctx = setup()
ctx.conditions


## Resumo por condição


In [ ]:
cols = ['V', 'E', 'F', 'C', 'genus', 'face_length_max', 'n_leaf_faces', 'n_bigon_faces']
cols = [c for c in cols if c in ctx.graph.columns]
ctx.graph.groupby('condition', observed=True)[cols].sum()


## Verificação da fórmula de Euler

`χ = V − E + F` tem de bater com `2C − 2g` em **toda** linha. Se alguma
falhar, há bug — não arredondamento.


In [ ]:
g = ctx.graph.copy()
if g.empty:
    print('sem estatísticas de grafo')
else:
    g['chi'] = g.V - g.E + g.F
    g['2C-2g'] = 2 * g.C - 2 * g.genus
    g['ok'] = g.chi == g['2C-2g']
    print('linhas consistentes:', int(g.ok.sum()), '/', len(g))
    display(g[['condition', 'sample_id', 'V', 'E', 'F', 'C', 'genus', 'chi', '2C-2g', 'ok']])
    assert g.ok.all(), 'fórmula de Euler violada — investigar'


## Distribuição de comprimento de faces (C9)

Se a curva estiver concentrada em uns poucos pontos de comprimento enorme,
a 'face' deixou de ser uma narrativa e virou a memória inteira.


In [ ]:
plot_face_length_distribution(ctx); show()


In [ ]:
if not ctx.faces.empty:
    tot = ctx.faces.groupby('condition', observed=True)['count'].sum()
    mx  = ctx.faces.groupby('condition', observed=True)['length'].max()
    wavg = (ctx.faces.assign(w=ctx.faces.length * ctx.faces['count'])
            .groupby('condition', observed=True)
            .apply(lambda d: d.w.sum() / d['count'].sum(), include_groups=False))
    display(pd.DataFrame({'faces': tot, 'maior face': mx,
                          'comprimento médio (ponderado)': wavg.round(1)}))


## Faces-folha vs bígonos genuínos (C3)

Uma face de comprimento 2 pode ser duas arestas paralelas (candidata a
redundância) **ou** uma aresta percorrida duas vezes (folha — nada a
colapsar). Só a primeira alimenta H3.


In [ ]:
if {'n_leaf_faces', 'n_bigon_faces'} <= set(ctx.graph.columns):
    d = ctx.graph.groupby('condition', observed=True)[['n_leaf_faces', 'n_bigon_faces']].sum()
    ax = d.plot(kind='bar', figsize=(8, 3.4), color=['#999999', '#0072B2'])
    ax.set_ylabel('nº de faces'); ax.set_xlabel('')
    ax.set_title('Faces de comprimento 2, separadas por tipo')
    ax.legend(['folha (não colapsável)', 'bígono genuíno'], fontsize=8)
    plt.setp(ax.get_xticklabels(), rotation=15, ha='right'); show()
    display(d)


## Crescimento sessão a sessão


In [ ]:
plot_graph_growth(ctx); show()


## Efeito da curadoria e da consolidação


In [ ]:
cols = [c for c in ['ingest_n_facts', 'ingest_n_edges', 'ingest_n_collapses',
                    'ingest_n_consolidations', 'ingest_n_incongruent',
                    'ingest_n_skipped_self_loops'] if c in ctx.graph.columns]
if cols:
    display(ctx.graph.groupby('condition', observed=True)[cols].sum())


## Topologia explica o F1?

Uma conversa por ponto. Poucos pontos — trate como exploratório, não como
evidência.


In [ ]:
if not ctx.graph.empty and 'f1' in ctx.graph.columns:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    for ax, x in zip(axes, ['F', 'genus', 'face_length_max']):
        if x not in ctx.graph.columns: continue
        for cond, grp in ctx.graph.groupby('condition', observed=True):
            ax.scatter(grp[x], grp.f1, label=cond, color=PALETTE.get(cond), s=28)
        ax.set_xlabel(x); ax.set_ylabel('F1 da conversa')
    axes[-1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
    show()
